# <span style="color: #1F1DB5;">SPRINT 17 – NLP y Textos

# <span style="color: #1F1DB5;">Notebook 2 – Embeddings, Modelos Pre-entrenados e Introducción a LLMs

## Ficha de la sesión

| Campo | Detalle |
|---|---|
| Programa | Data Science |
| Sprint / sesión | Sprint 17 · Clase 2 |
| Tema | Embeddings y LLMs |
| Audiencia | Tutor |
| Tipo de sesión | teórico-práctica |
| Duración de referencia | 120 minutos, incluida práctica, validación y cierre |
| Última actualización | 2026-08-19 |



## Prerrequisitos y preparación

- **Conceptuales:** Python, scikit-learn, vectores y evaluación básica de modelos.
- **Técnicos:** Python 3.11 o entorno compatible; instalar dependencias desde `../../requirements.txt`.
- **Archivos y red:** usa solo rutas relativas. Cuando haya una operación externa, revisa su nota de contingencia antes de clase.
- **Reproducibilidad:** usa la semilla de referencia `42` en cualquier operación aleatoria y documenta excepciones.


## Resultado esperado y evidencia

**Entregable:** Comparación local de representaciones y diseño seguro de un caso opcional con LLM.

**Criterio de aceptación principal:** No incluye secretos; usa OPENAI_API_KEY solo como variable de entorno y mantiene alternativa sin API.

La evidencia debe quedar dentro del notebook: código o procedimiento, comprobación del resultado y una interpretación breve.


## Ruta de trabajo e instrucciones de uso

1. Lee la pregunta guía, los objetivos y el entregable antes de ejecutar código.
2. Ejecuta las celdas en orden; si cambias una definición, reinicia el kernel y usa **Run All**.
3. En cada actividad sigue: **propósito → intento → validación → interpretación**.
4. Ejecuta primero en kernel limpio; guía la discusión antes de mostrar soluciones o criterios esperados.
5. Registra cualquier supuesto, fuente externa o cambio de datos.

> **Control obligatorio de esta clase:** No incluye secretos; usa OPENAI_API_KEY solo como variable de entorno y mantiene alternativa sin API.


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Comprender qué son los **embeddings** y por qué superan a BoW/TF-IDF.
- Entender la intuición de **Word2Vec** (king - man + woman = queen).
- Diferenciar entre representaciones **sparse** (BoW) y **dense** (embeddings).
- Conocer la arquitectura **Transformer** y por qué revolucionó NLP.
- Entender cómo se usan **LLMs** (GPT, BERT) en la práctica de Data Science.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
De BoW a Embeddings | 20 min |
Word2Vec y representaciones densas | 20 min |
Modelos pre-entrenados y Transformers | 20 min |
LLMs en la práctica | 20 min |
Debate: BoW vs Embeddings | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Cómo lograr que king - man + woman = queen?

En la clase anterior, convertimos texto en números con BoW y TF-IDF. Pero esas representaciones no capturan **significado**: "feliz" y "contento" tienen vectores completamente diferentes aunque signifiquen lo mismo. Los embeddings resuelven esto: palabras con significado similar tienen vectores cercanos en el espacio.

## <span style="color: #2826DE;">De BoW a Embeddings: El Problema de las Representaciones Sparse (20 mins)

**Limitaciones de BoW/TF-IDF:**

1. **No capturan semántica:** "perro" y "can" tienen vectores completamente diferentes.
2. **No capturan orden:** "el perro mordió al hombre" y "el hombre mordió al perro" tienen la misma representación.
3. **Alta dimensionalidad:** El vector tiene tantas dimensiones como palabras en el vocabulario (miles o millones).
4. **Sparse:** La mayoría de valores son 0 (un documento usa pocas palabras del vocabulario total).

**La solución: Embeddings (representaciones densas)**

| Propiedad | BoW/TF-IDF | Embeddings |
|---|---|---|
| Dimensionalidad | Miles/millones | 50-300 típicamente |
| Tipo | Sparse (mayormente 0s) | Dense (todos los valores informativos) |
| Semántica | No captura significado | Palabras similares → vectores cercanos |
| Relaciones | No las modela | Captura analogías y relaciones |
| Entrenamiento | Solo cuenta frecuencias | Aprende de contexto |

**Idea clave:** En lugar de un vector gigante con conteos, cada palabra se representa con un vector denso de ~100-300 números que CAPTURA su significado basándose en los contextos donde aparece.

Visualicemos la diferencia entre representaciones sparse y dense con un ejemplo sencillo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ejemplo: similitud semántica con TF-IDF vs embeddings simulados
palabras = ["rey", "reina", "hombre", "mujer", "niño", "niña", "perro", "gato"]

# TF-IDF no captura semántica (simulación del problema)
# Cada palabra es un "documento" de una sola palabra
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(palabras)
sim_tfidf = cosine_similarity(tfidf_matrix)

print("Similitud con TF-IDF (todas las palabras son 'diferentes'):")
print(f"  rey - reina:   {sim_tfidf[0,1]:.3f}")  # Deberían ser similares
print(f"  hombre - mujer: {sim_tfidf[2,3]:.3f}")  # Deberían ser similares
print(f"  rey - perro:   {sim_tfidf[0,6]:.3f}")  # Deberían ser diferentes

# Embeddings simulados (capturan semántica)
# Dimensiones: [realeza, género, edad, animal]
embeddings = {
    "rey":    np.array([0.9, 0.8, 0.5, 0.0]),
    "reina":  np.array([0.9, 0.2, 0.5, 0.0]),
    "hombre": np.array([0.1, 0.8, 0.5, 0.0]),
    "mujer":  np.array([0.1, 0.2, 0.5, 0.0]),
    "niño":   np.array([0.1, 0.8, 0.1, 0.0]),
    "niña":   np.array([0.1, 0.2, 0.1, 0.0]),
    "perro":  np.array([0.0, 0.5, 0.3, 0.9]),
    "gato":   np.array([0.0, 0.5, 0.3, 0.8]),
}

emb_matrix = np.array([embeddings[p] for p in palabras])
sim_emb = cosine_similarity(emb_matrix)

print("\nSimilitud con Embeddings (capturan semántica):")
print(f"  rey - reina:    {sim_emb[0,1]:.3f}")
print(f"  hombre - mujer: {sim_emb[2,3]:.3f}")
print(f"  perro - gato:   {sim_emb[6,7]:.3f}")
print(f"  rey - perro:    {sim_emb[0,6]:.3f}")

# La magia: analogías
result = embeddings["rey"] - embeddings["hombre"] + embeddings["mujer"]
print(f"\n✨ rey - hombre + mujer = {result}")
print(f"   Vector de 'reina':     {embeddings['reina']}")
print(f"   ¡Son prácticamente iguales!")

Preguntas:

- ¿Por qué TF-IDF da similitud 0 entre "rey" y "reina" mientras que embeddings sí capturan la relación?

- ¿Qué representa cada dimensión en un embedding? ¿Son interpretables como en nuestro ejemplo simplificado?

## <span style="color: #2826DE;">Word2Vec y Representaciones Densas (20 mins)

**Word2Vec** (Google, 2013) fue el primer modelo exitoso de embeddings a gran escala. Su idea central es brillante en su simplicidad:

> "Dime con quién andas y te diré quién eres" — pero para palabras.

**Hipótesis distribucional:** Palabras que aparecen en contextos similares tienen significados similares.
- "El **gato** se sentó en el sofá" / "El **perro** se sentó en el sofá" → gato ≈ perro

**Dos arquitecturas:**
1. **CBOW (Continuous Bag of Words):** Predice la palabra central dado su contexto.
   - Contexto: "el _ se sentó" → Predice: "gato"
2. **Skip-gram:** Predice el contexto dada la palabra central.
   - Palabra: "gato" → Predice: "el", "se", "sentó"

**Resultado:** Cada palabra tiene un vector de ~100-300 dimensiones que codifica su significado.

**Propiedades mágicas:**
- `king - man + woman ≈ queen` (analogía de género)
- `Paris - France + Italy ≈ Rome` (analogía de capital)
- `bigger - big + small ≈ smaller` (analogía gramatical)

**Similitud coseno:** Para medir qué tan "cercanas" son dos palabras en el espacio de embeddings.

Veamos cómo entrenar Word2Vec con gensim en un corpus pequeño y explorar las relaciones que aprende.

In [ ]:
from gensim.models import Word2Vec

# Corpus de ejemplo (en producción usarías millones de oraciones)
corpus = [
    ["el", "rey", "gobierna", "el", "reino", "con", "sabiduría"],
    ["la", "reina", "gobierna", "el", "reino", "con", "gracia"],
    ["el", "hombre", "trabaja", "en", "la", "oficina"],
    ["la", "mujer", "trabaja", "en", "la", "oficina"],
    ["el", "niño", "juega", "en", "el", "parque"],
    ["la", "niña", "juega", "en", "el", "parque"],
    ["el", "perro", "corre", "por", "el", "parque"],
    ["el", "gato", "duerme", "en", "el", "sofá"],
    ["el", "rey", "tiene", "una", "corona", "de", "oro"],
    ["la", "reina", "tiene", "una", "corona", "de", "plata"],
    ["el", "hombre", "es", "fuerte", "y", "valiente"],
    ["la", "mujer", "es", "fuerte", "e", "inteligente"],
    ["el", "príncipe", "es", "hijo", "del", "rey"],
    ["la", "princesa", "es", "hija", "de", "la", "reina"],
] * 100  # Repetimos para tener más datos

# Entrenar Word2Vec
model_w2v = Word2Vec(
    sentences=corpus,
    vector_size=50,    # Dimensiones del embedding
    window=5,          # Contexto: 5 palabras a cada lado
    min_count=2,       # Ignorar palabras con < 2 ocurrencias
    workers=4,
    epochs=100,
    seed=42
)

# Explorar similitudes
print("Palabras más similares a 'rey':")
for word, sim in model_w2v.wv.most_similar('rey', topn=5):
    print(f"  {word}: {sim:.3f}")

print("\nPalabras más similares a 'mujer':")
for word, sim in model_w2v.wv.most_similar('mujer', topn=5):
    print(f"  {word}: {sim:.3f}")

# Analogía: rey - hombre + mujer = ?
print("\n✨ Analogía: rey - hombre + mujer =")
try:
    result = model_w2v.wv.most_similar(positive=['rey', 'mujer'], negative=['hombre'], topn=3)
    for word, sim in result:
        print(f"  {word}: {sim:.3f}")
except:
    print("  (corpus muy pequeño para analogías perfectas)")

print(f"\nDimensiones de cada embedding: {model_w2v.wv.vector_size}")
print(f"Vocabulario: {len(model_w2v.wv)} palabras")

## <span style="color: #2826DE;">Modelos Pre-entrenados y Transformers (20 mins)

**El problema de Word2Vec:** Cada palabra tiene UN solo vector, sin importar el contexto.
- "Banco" (financiero) y "banco" (para sentarse) → ¡mismo vector!
- No captura que "manzana" significa cosas diferentes en "comí una manzana" vs "tengo un iPhone de Apple (manzana)".

**La revolución Transformer (2017):**
El paper "Attention Is All You Need" introdujo una arquitectura que cambió todo NLP:

1. **Atención (Attention):** El modelo "mira" todas las palabras de la oración al procesar cada una, decidiendo cuáles son relevantes para el contexto actual.
2. **Contextual embeddings:** El vector de cada palabra CAMBIA según su contexto.
3. **Paralelización:** A diferencia de RNNs, procesa toda la secuencia a la vez.

**Modelos pre-entrenados famosos:**

| Modelo | Creador | Uso principal |
|---|---|---|
| BERT | Google | Entender texto (clasificación, NER, Q&A) |
| GPT-4 | OpenAI | Generar texto (chat, resumen, código) |
| Claude | Anthropic | Asistente conversacional |
| T5 | Google | Texto-a-texto (traducción, resumen) |

**Transfer Learning para NLP:**
1. Alguien entrena un modelo ENORME con todo internet (cuesta millones de USD).
2. Tú tomas ese modelo pre-entrenado y lo **afinas** (fine-tune) con tus datos específicos.
3. Con pocos datos propios, obtienes resultados excelentes.

Es como contratar a alguien que ya sabe leer y solo necesita aprender tu dominio específico.

Preguntas:

- ¿Por qué los embeddings contextuales (BERT) son mejores que Word2Vec para palabras polisémicas?

- ¿Cuál es la ventaja de usar transfer learning en vez de entrenar un modelo desde cero?

- ¿Qué tipo de tareas resuelve mejor BERT vs GPT?

## <span style="color: #2826DE;">LLMs en la Práctica: APIs y Casos de Uso (20 mins)

Los **Large Language Models** (LLMs) son modelos Transformer entrenados con cantidades masivas de texto. Son tan grandes que no los entrenamos ni ejecutamos localmente: los usamos vía **APIs**.

**Casos de uso para Data Scientists:**
- **Clasificación de texto:** Clasificar tickets de soporte, sentimiento, temas.
- **Extracción de información:** Sacar entidades, fechas, montos de texto no estructurado.
- **Resumen:** Condensar documentos largos en puntos clave.
- **Generación de datos:** Crear datos sintéticos para entrenamiento.
- **Augmentation:** Parafrasear textos para aumentar el dataset.
- **Code generation:** Generar código Python para análisis.

**Cómo se usan en la práctica:**
```python
# Pseudocódigo de uso con API
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "Clasifica el sentimiento como positivo o negativo"},
        {"role": "user", "content": "La comida estuvo deliciosa"}
    ]
)
# response → "positivo"
```

**Ventajas de LLMs:**
- Zero-shot: Funcionan sin datos de entrenamiento propios.
- Flexibles: Una misma API para múltiples tareas.
- Excelente calidad en tareas de lenguaje.

**Limitaciones:**
- Costo por token (pueden ser caros a escala).
- Latencia (más lentos que modelos locales).
- Alucinaciones (pueden inventar información).
- Privacidad (tus datos van a un servidor externo).

#### Criterios de logro

- La respuesta atiende todos los elementos de la consigna.
- El resultado incluye una comprobación observable, no solo código o una afirmación.
- La conclusión explica qué significa el resultado y menciona al menos un supuesto.
- **Control específico:** No incluye secretos; usa OPENAI_API_KEY solo como variable de entorno y mantiene alternativa sin API.

> **Nota para tutor:** pide una predicción antes de ejecutar, valida el criterio en vivo y comenta al menos un error frecuente.


Veamos un ejemplo conceptual de cómo integrar un LLM en un pipeline de Data Science para clasificación masiva.

In [ ]:
# Ejemplo conceptual: cuándo usar cada enfoque
import pandas as pd

comparacion = pd.DataFrame({
    'Criterio': ['Datos etiquetados', 'Costo', 'Velocidad', 'Calidad', 'Privacidad', 'Personalización'],
    'BoW + LogReg': ['Necesita muchos', 'Gratis', 'Muy rápido', 'Buena con datos', 'Total (local)', 'Alta'],
    'Embeddings + ML': ['Necesita algunos', 'Bajo', 'Rápido', 'Muy buena', 'Total (local)', 'Alta'],
    'LLM (API)': ['No necesita (zero-shot)', 'Por uso', 'Lento', 'Excelente', 'Baja (nube)', 'Limitada']
})

print("¿Cuándo usar cada enfoque?")
print(comparacion.to_string(index=False))

print("""
\n📋 Recomendaciones prácticas:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Tienes MUCHOS datos etiquetados → TF-IDF + LogisticRegression
• Tienes POCOS datos etiquetados  → Embeddings pre-entrenados + ML
• No tienes datos etiquetados     → LLM vía API (zero-shot)
• Necesitas PRIVACIDAD total      → Modelos locales (BoW/embeddings)
• Necesitas máxima CALIDAD        → LLM fine-tuned o embeddings BERT
""")

## <span style="color: #2826DE;">Debate: BoW vs Embeddings (15 mins)

**Dinámica:** La clase se divide en dos equipos:

🔵 **Equipo BoW/TF-IDF:** Defiende que las representaciones clásicas siguen siendo la mejor opción.

🔴 **Equipo Embeddings/LLMs:** Argumenta que los embeddings y LLMs son superiores en todo.

**Reglas del debate:**
1. Cada equipo tiene 3 minutos para preparar sus argumentos.
2. Equipo BoW presenta primero (2 min).
3. Equipo Embeddings responde (2 min).
4. Réplicas cruzadas (3 min).
5. El instructor modera y cierra con la conclusión real.

**Puntos a considerar:**
- Interpretabilidad
- Costo computacional
- Necesidad de datos
- Privacidad
- Escalabilidad
- Calidad de resultados

**Spoiler:** No hay ganador absoluto. La mejor herramienta depende del contexto. Un buen Data Scientist sabe cuándo usar cada una.

## <span style="color: #2826DE;">Tips y buenas prácticas

- **Empieza simple:** BoW/TF-IDF primero como baseline. Solo usa embeddings si necesitas más calidad.
- Los embeddings pre-entrenados (Word2Vec, GloVe, FastText) son gratuitos y excelentes para muchas tareas.
- Para tareas en español, busca embeddings entrenados en español (no traduzcas del inglés).
- Los LLMs son herramientas, no magia. Valida sus resultados como cualquier otro modelo.
- Fine-tuning de BERT es más eficiente que entrenar desde cero para dominios específicos.
- Considera el costo: una API de LLM puede ser cara si procesas millones de textos.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Qué problema de BoW/TF-IDF resuelven los embeddings?

- La velocidad de entrenamiento
- La captura de significado semántico ✅
- El tamaño del dataset
- La privacidad de los datos


2️⃣ ¿Qué resultado produce la analogía king - man + woman en Word2Vec?

- king
- man
- queen ✅
- woman


3️⃣ ¿Cuál es la idea central de Word2Vec?

- Palabras frecuentes son más importantes
- Palabras en contextos similares tienen significados similares ✅
- Las palabras largas son más informativas
- El orden de las palabras no importa


4️⃣ ¿Qué ventaja tienen los embeddings contextuales (BERT) sobre Word2Vec?

- Son más rápidos
- El vector de una palabra cambia según el contexto ✅
- Usan menos memoria
- No necesitan entrenamiento


5️⃣ ¿Cuándo es mejor usar un LLM vía API que TF-IDF + LogisticRegression?

- Cuando tienes millones de datos etiquetados
- Cuando no tienes datos etiquetados (zero-shot) ✅
- Cuando necesitas máxima privacidad
- Cuando el presupuesto es muy limitado


6️⃣ ¿Qué es transfer learning en NLP?

- Traducir de un idioma a otro
- Usar un modelo pre-entrenado y afinarlo con tus datos ✅
- Transferir datos entre servidores
- Copiar los pesos de un modelo a otro sin cambios

## <span style="color: #2826DE;">Visualización de Embeddings en 2D

Los embeddings viven en un espacio de muchas dimensiones (50-300). Podemos usar PCA o t-SNE para proyectarlos a 2D y ver si las relaciones semánticas se mantienen visualmente.

In [ ]:
from sklearn.decomposition import PCA

# Obtener todos los vectores del modelo Word2Vec
words = list(model_w2v.wv.key_to_index.keys())
vectors = np.array([model_w2v.wv[w] for w in words])

# Reducir a 2D con PCA
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

# Visualizar
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.3, s=20, color='gray')

# Destacar palabras interesantes
highlight = ['rey', 'reina', 'hombre', 'mujer', 'niño', 'niña', 'perro', 'gato']
for word in highlight:
    if word in words:
        idx = words.index(word)
        ax.scatter(vectors_2d[idx, 0], vectors_2d[idx, 1], s=100, zorder=5)
        ax.annotate(word, (vectors_2d[idx, 0], vectors_2d[idx, 1]),
                   fontsize=12, fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')

ax.set_title('Embeddings Word2Vec proyectados a 2D (PCA)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

print("Observa cómo palabras semánticamente similares")
print("aparecen cercanas en el espacio 2D.")

Preguntas:

- ¿Por qué palabras como "rey" y "reina" aparecen cercanas pero no idénticas?

- ¿Qué información se pierde al proyectar de 50 dimensiones a 2?

- ¿Cómo usarías estos embeddings como features para un modelo de clasificación?

## <span style="color: #2826DE;">Embeddings como Features para Clasificación

Una forma práctica de usar embeddings para clasificación de documentos es promediar los vectores de todas las palabras del texto. Esto produce un vector de longitud fija que puede alimentar cualquier clasificador.

In [ ]:
# Ejemplo: usar embeddings como features para clasificación
def documento_a_vector(texto, modelo, dim=50):
    """Convierte un texto en un vector promediando los embeddings de sus palabras."""
    palabras = texto.lower().split()
    vectores = []
    for p in palabras:
        if p in modelo.wv:
            vectores.append(modelo.wv[p])
    if vectores:
        return np.mean(vectores, axis=0)
    else:
        return np.zeros(dim)

# Ejemplo con textos simples
textos = [
    "el rey gobierna el reino",
    "la reina gobierna con gracia",
    "el perro corre por el parque",
    "el gato duerme en el sofá"
]

print("Vectores de documentos (primeros 5 dims):")
for texto in textos:
    vec = documento_a_vector(texto, model_w2v)
    print(f"  '{texto}' → [{', '.join(f'{v:.3f}' for v in vec[:5])}...]")

# Similitud entre documentos
from sklearn.metrics.pairwise import cosine_similarity

doc_vecs = np.array([documento_a_vector(t, model_w2v) for t in textos])
sim_matrix = cosine_similarity(doc_vecs)

print("\nSimilitud coseno entre documentos:")
for i in range(len(textos)):
    for j in range(i+1, len(textos)):
        print(f"  '{textos[i][:30]}' vs '{textos[j][:30]}': {sim_matrix[i,j]:.3f}")

## Cierre, límites y siguientes pasos

### Síntesis

La clase de **Embeddings y LLMs** conecta los conceptos con una evidencia reproducible. El trabajo se considera completo cuando produce **comparación local de representaciones y diseño seguro de un caso opcional con LLM.** y satisface el control técnico definido al inicio.

### Limitaciones que deben declararse

- Los resultados dependen del alcance y la calidad de los datos o ejemplos usados.
- Una ejecución correcta no demuestra por sí sola que el resultado generalice a otros contextos.
- Las decisiones de modelado, limpieza o comunicación deben justificarse según el problema.

### Lista de salida

- [ ] Respondí la pregunta guía con evidencia.
- [ ] Validé el resultado con el criterio de logro: No incluye secretos; usa OPENAI_API_KEY solo como variable de entorno y mantiene alternativa sin API.
- [ ] Documenté supuestos, límites y decisiones.
- [ ] Dejé rutas portables, sin credenciales ni información privada.

Antes de publicar, usa Restart Kernel and Run All y conserva únicamente outputs breves que ayuden a verificar el resultado esperado.

### Ayuda y recursos

- Revisa los ejemplos y el glosario de esta clase antes de cambiar el enfoque.
- Lleva dudas concretas y el mensaje de error completo a los canales de soporte u office hours.
- Continúa con la siguiente clase del sprint y reutiliza este entregable como evidencia de aprendizaje.
